# Ferni Tool Router - Qwen 2.5 0.5B (v3 - Fixed)

**Fixes:** float32 for stable training, lower LR, proper gradient clipping

In [ ]:
!pip install -q transformers datasets accelerate peft safetensors

In [ ]:
from google.colab import files
print("Upload: train.jsonl, validation.jsonl, label_map.json")
uploaded = files.upload()

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
assert torch.cuda.is_available(), "Need GPU!"

In [ ]:
import json

with open('label_map.json') as f:
    label_map = json.load(f)
num_labels = len(label_map)
print(f"{num_labels} labels")

train_data = [json.loads(l) for l in open('train.jsonl')]
val_data = [json.loads(l) for l in open('validation.jsonl')]
print(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

MODEL = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=num_labels,
    trust_remote_code=True,
    torch_dtype=torch.float32,
)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["score"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.cuda()
print("Model ready (float32)")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np

class ToolDataset(Dataset):
    def __init__(self, data, tokenizer, label_map):
        self.data = data
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.num_labels = len(label_map)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        query = item['query']
        tools = item.get('selected_tools', item.get('tools', []))
        if isinstance(tools, str): tools = [tools]

        enc = self.tokenizer(query, truncation=True, max_length=64, padding='max_length', return_tensors='pt')
        
        labels = np.zeros(self.num_labels, dtype=np.float32)
        for t in tools:
            if t in self.label_map:
                labels[self.label_map[t]] = 1.0

        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': torch.tensor(labels, dtype=torch.float32)
        }

train_ds = ToolDataset(train_data, tokenizer, label_map)
val_ds = ToolDataset(val_data, tokenizer, label_map)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
print(f"{len(train_loader)} train batches")

In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm
import torch.nn as nn

EPOCHS = 3
LR = 5e-5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
loss_fn = nn.BCEWithLogitsLoss()

print(f"Training {EPOCHS} epochs @ lr={LR}")
history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in pbar:
        optimizer.zero_grad()
        
        ids = batch['input_ids'].cuda()
        mask = batch['attention_mask'].cuda()
        labels = batch['labels'].cuda()

        out = model(input_ids=ids, attention_mask=mask)
        loss = loss_fn(out.logits, labels)
        
        if torch.isnan(loss):
            print("NaN loss, skipping")
            continue
            
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].cuda()
            mask = batch['attention_mask'].cuda()
            labels = batch['labels'].cuda()
            
            out = model(input_ids=ids, attention_mask=mask)
            preds = out.logits.argmax(dim=1)
            
            for i in range(len(preds)):
                if labels[i, preds[i]] > 0.5:
                    correct += 1
                total += 1
    
    acc = correct / total
    avg_loss = total_loss / len(train_loader)
    history.append({'epoch': epoch+1, 'loss': avg_loss, 'acc': acc})
    print(f"Epoch {epoch+1}: loss={avg_loss:.4f}, acc={acc:.1%}")

print(f"Done! Final acc: {acc:.1%}")

In [ ]:
id_to_label = {v: k for k, v in label_map.items()}

tests = ["play jazz", "weather", "set timer 5 min", "turn off lights", "feeling stressed"]

model.eval()
for q in tests:
    enc = tokenizer(q, return_tensors='pt', truncation=True, max_length=64).to('cuda')
    with torch.no_grad():
        out = model(**enc)
        probs = torch.sigmoid(out.logits[0])
        top = torch.topk(probs, 3)
    
    print(f"'{q}'")
    for s, i in zip(top.values.tolist(), top.indices.tolist()):
        print(f"  {id_to_label[i]}: {s:.1%}")

In [ ]:
import shutil, json

model.save_pretrained('output')
tokenizer.save_pretrained('output')
shutil.copy('label_map.json', 'output/')
with open('output/history.json', 'w') as f:
    json.dump(history, f)

!zip -r ferni-router-v3.zip output/
files.download('ferni-router-v3.zip')